# Notebook 04 — Grad-CAM & Attention Visualization

What does the fine-tuned ViT actually look at when classifying a scene?

Three complementary views:
- **Part 1 — Grad-CAM:** Class-discriminative heatmap via gradients through the last transformer block
- **Part 2 — Attention maps:** Raw CLS-token attention weights from the last block, reshaped 14×14
- **Part 3 — Attention rollout:** Propagate attention across all 12 layers for a full-image view

---

## Table of Contents

1. [Setup & Load Model](#part-1)
2. [Grad-CAM](#part-2)
3. [Attention Maps (Last Block)](#part-3)
4. [Attention Rollout](#part-4)
5. [Export to Assets](#part-5)

---

In [ ]:

# ── Colab setup ───────────────────────────────────────────────────────────────
# No-op when running locally.
# On Colab: mounts Drive, installs packages, sets up paths so ../models etc. work.
# Run notebook 03 first — this notebook loads the checkpoint it saves.
import sys, os

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import subprocess
    subprocess.run(['pip', 'install', '-q', 'timm==1.0.25', 'einops==0.8.2'], check=True)

    DRIVE_BASE = '/content/drive/MyDrive/app-01'
    for d in ['models', 'assets', 'data']:
        os.makedirs(f'{DRIVE_BASE}/{d}', exist_ok=True)

    os.makedirs('/content/app-01/notebooks', exist_ok=True)
    for d in ['models', 'assets', 'data']:
        dst = f'/content/app-01/{d}'
        if not os.path.exists(dst):
            os.symlink(f'{DRIVE_BASE}/{d}', dst)

    os.chdir('/content/app-01/notebooks')
    print(f'Colab setup complete. CWD: {os.getcwd()}')

    # Verify checkpoint exists
    ckpt = f'{DRIVE_BASE}/models/vit_finetuned_sun397.pth'
    if os.path.exists(ckpt):
        print(f'Checkpoint found: {ckpt}')
    else:
        print('WARNING: checkpoint not found — run notebook 03 first.')
else:
    print('Local environment detected.')
# ─────────────────────────────────────────────────────────────────────────────


<a id="part-1"></a>
## Part 1 — Setup & Load Model

In [ ]:
import torch
import timm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
from torchvision import transforms, datasets
from pathlib import Path
import random

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

CLASSES = ['beach', 'forest', 'mountain', 'kitchen', 'bedroom',
           'street', 'restaurant', 'office', 'living_room', 'park']

CHECKPOINT = Path('../models/vit_finetuned_sun397.pth')
ASSETS_DIR = Path('../assets')
ASSETS_DIR.mkdir(exist_ok=True)

In [ ]:
# Load fine-tuned ViT
model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=10)

if CHECKPOINT.exists():
    state = torch.load(CHECKPOINT, map_location='cpu')
    model.load_state_dict(state)
    print(f'Loaded checkpoint: {CHECKPOINT}')
else:
    print(f'WARNING: checkpoint not found at {CHECKPOINT}')
    print('Run notebook 03 first to train and save the model.')

model = model.to(device)
model.eval()
print('Model ready.')

In [ ]:
from datasets import load_dataset

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

TARGET_CLASSES = ['beach', 'forest', 'mountain', 'kitchen', 'bedroom',
                  'street', 'restaurant', 'office', 'living room', 'park']
TARGET_INDICES = [46, 159, 244, 213, 48, 335, 300, 255, 227, 265]
idx_to_new = {old: new for new, old in enumerate(TARGET_INDICES)}

hf_test = load_dataset('pc-ml-dl/sun397', split='test')
print(f'Loaded {len(hf_test):,} test images')

In [ ]:
# Pick one sample image per class
import random
random.seed(42)

by_class = {i: None for i in range(10)}
for row in hf_test:
    if row['label'] in idx_to_new:
        new_label = idx_to_new[row['label']]
        if by_class[new_label] is None:
            by_class[new_label] = row['image'].convert('RGB')
    if all(v is not None for v in by_class.values()):
        break

sample_images = [(img, TARGET_CLASSES[i], i) for i, img in by_class.items()]
print(f'Loaded {len(sample_images)} sample images (one per class)')

[↑ Back to top](#part-1)

---

<a id="part-2"></a>
## Part 2 — Grad-CAM

**What Grad-CAM does:**  
Computes the gradient of the predicted class score with respect to the activations of the last transformer block. Spatial positions where the gradient is large and positive are the ones most responsible for that class prediction — the model's "evidence".

**ViT adaptation:**  
CNN Grad-CAM targets the last conv feature map. In ViT, the equivalent is the output of `model.blocks[-1]` — a (B, 197, 768) tensor, where 197 = 1 CLS token + 196 patch tokens. We discard CLS and use the 196 patch activations, reshaped to 14×14.

In [ ]:
import math

class AttentionCAM:
    """
    Attention-based saliency for ViT.
    
    ViT classifies via the CLS token, not patch tokens — gradients don't flow
    meaningfully to individual patches. Instead, we extract the attention weights
    from the last transformer block to see which patches the CLS token attended to
    when making its prediction.
    
    Hook: QKV projection in the last block → reconstruct attention matrix manually.
    Output: 14×14 map of CLS-to-patch attention, averaged across 12 heads.
    """
    def __init__(self, model):
        self.model = model
        self._hooks = []
        self._captured = {}

    def _hook_fn(self, m, inp, out):
        self._captured['qkv'] = out.detach()

    def register_hooks(self):
        h = self.model.blocks[-1].attn.qkv.register_forward_hook(self._hook_fn)
        self._hooks.append(h)

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()
        self._hooks = []

    def __call__(self, img_tensor):
        """
        Args:
            img_tensor: (1, 3, 224, 224) on device
        Returns:
            cam: (14, 14) numpy float32, normalized 0–1
            pred_class: predicted class index
            pred_prob: predicted class probability
        """
        self._captured = {}
        self.register_hooks()
        with torch.no_grad():
            logits = self.model(img_tensor)
        self.remove_hooks()

        probs = torch.softmax(logits, dim=-1)
        pred_class = logits.argmax(dim=-1).item()
        pred_prob = probs[0, pred_class].item()

        # Reconstruct attention weights from QKV
        qkv = self._captured['qkv']          # (1, N, 3*dim)
        B, N, _ = qkv.shape
        num_heads = self.model.blocks[-1].attn.num_heads
        head_dim = qkv.shape[-1] // (3 * num_heads)
        qkv = qkv.reshape(B, N, 3, num_heads, head_dim).permute(2, 0, 3, 1, 4)
        q, k = qkv[0], qkv[1]               # each (1, H, N, head_dim)
        scale = math.sqrt(head_dim)
        attn = torch.softmax(q @ k.transpose(-2, -1) / scale, dim=-1)  # (1, H, N, N)

        # CLS token (index 0) attending to each patch (indices 1:)
        cls_attn = attn[0, :, 0, 1:].mean(0)  # (196,) — mean over heads
        cam = cls_attn.reshape(14, 14).cpu().numpy().astype('float32')
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, pred_class, pred_prob

print("AttentionCAM class defined.")
print("Note: ViT classifies via the CLS token — gradients don't flow to patch tokens.")
print("Attention weights show where the CLS token looked when making its prediction.")


In [ ]:
def overlay_cam(pil_img, cam, alpha=0.5, colormap='jet'):
    """Overlay a 14x14 CAM heatmap on a PIL image. Returns a numpy array."""
    import cv2
    img_np = np.array(pil_img)          # (224, 224, 3)
    h, w = img_np.shape[:2]

    # Upsample cam to image size
    cam_resized = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize(
        (w, h), Image.BILINEAR)) / 255.0

    # Apply colormap
    cmap = cm.get_cmap(colormap)
    heatmap = cmap(cam_resized)[:, :, :3]   # drop alpha channel
    heatmap = (heatmap * 255).astype(np.uint8)

    # Blend
    overlay = (alpha * heatmap + (1 - alpha) * img_np).astype(np.uint8)
    return overlay

print('overlay_cam defined.')

In [ ]:
# Run attention maps on all 10 sample images
# Apply full preprocessing (resize + crop) to match training transform
resize_crop = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
])
to_tensor = transforms.ToTensor()
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225])

gradcam = AttentionCAM(model)

fig, axes = plt.subplots(len(sample_images), 3, figsize=(12, 4 * len(sample_images)))

if len(sample_images) == 1:
    axes = [axes]

for row, (pil_img, true_class, true_idx) in enumerate(sample_images):
    pil_224 = resize_crop(pil_img)
    img_t = normalize(to_tensor(pil_224)).unsqueeze(0).to(device)
    cam, pred_idx, pred_prob = gradcam(img_t)

    pred_class = CLASSES[pred_idx]
    correct = '✓' if pred_idx == true_idx else '✗'

    overlay = overlay_cam(pil_224, cam)

    # Column 0: original
    axes[row][0].imshow(pil_224)
    axes[row][0].set_title(f'True: {true_class}', fontsize=10)
    axes[row][0].axis('off')

    # Column 1: attention map (14×14 grid)
    axes[row][1].imshow(cam, cmap='jet', vmin=0, vmax=1)
    axes[row][1].set_title('CLS attention (last block)', fontsize=10)
    axes[row][1].axis('off')

    # Column 2: overlay
    axes[row][2].imshow(overlay)
    axes[row][2].set_title(f'Pred: {pred_class} {correct} ({pred_prob:.1%})', fontsize=10)
    axes[row][2].axis('off')

plt.suptitle('Attention Maps — Fine-tuned ViT on SUN397\n(CLS token attention from last block, mean over 12 heads)', fontsize=12)
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'gradcam.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved gradcam.png")


In [ ]:
# Assertions
gradcam_test = AttentionCAM(model)
test_img = sample_images[0][0]
test_t = normalize(to_tensor(test_img)).unsqueeze(0).to(device)
test_cam, test_pred, test_prob = gradcam_test(test_t)

assert test_cam.shape == (14, 14), f'Expected (14,14), got {test_cam.shape}'
assert 0.0 <= test_cam.min() and test_cam.max() <= 1.0, 'CAM values out of [0,1]'
assert 0 <= test_pred < 10, f'Pred class out of range: {test_pred}'
assert 0.0 < test_prob <= 1.0, f'Probability out of range: {test_prob}'

gradcam_test.remove_hooks()
print('All Grad-CAM assertions passed.')

[↑ Back to top](#part-1)

---

<a id="part-3"></a>
## Part 3 — Attention Maps (Last Block)

**What this shows:**  
The raw attention weights the CLS token places on each of the 196 patch positions in the final transformer block. High attention = the model is "looking at" that patch to make its prediction.

**Why the last block?**  
Earlier blocks attend to low-level structure (edges, textures). The last block attends to semantically meaningful regions — the features that actually feed into the classification head.

**Shape:**  `(B, num_heads, 197, 197)` → take row 0 (CLS token attends to patches) → average over heads → reshape to 14×14.

In [ ]:
class AttentionExtractor:
    """
    Extracts attention weights from every transformer block in a timm ViT.
    Uses forward hooks on each block's attention module.
    """
    def __init__(self, model):
        self.model = model
        self.attentions = []   # list of (B, H, 197, 197), one per block
        self._hooks = []
        self._register_hooks()

    def _register_hooks(self):
        for block in self.model.blocks:
            # timm ViT block: block.attn is the attention module
            # We need to hook the softmax output inside it.
            # Easiest: hook the block itself and re-run attn to get weights.
            # Better: patch the attn module's forward to also return weights.
            h = block.attn.register_forward_hook(self._attn_hook)
            self._hooks.append(h)

    def _attn_hook(self, module, input, output):
        # timm Attention.forward returns only the projected output (B, N, C)
        # We need to re-derive the weights. Store q, k from input.
        # Alternative: monkey-patch attn_drop to intercept the softmax.
        # We'll use the stored q, k via a separate approach below.
        pass

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()

# Note: timm's Attention module does not expose attention weights directly.
# We hook the QKV projection and reconstruct attention weights from Q and K manually.

def extract_attention_weights(model, img_tensor):
    """
    Extract attention weight matrices from all blocks of a timm ViT.

    Strategy: temporarily replace each block's attn_drop with an identity
    function that also saves the input (= the attention weights before dropout).

    Args:
        model: timm ViT, in eval mode
        img_tensor: (1, 3, 224, 224) on device

    Returns:
        attn_weights: list of (1, num_heads, 197, 197), one per block
    """
    attn_weights = []
    original_drops = []

    for block in model.blocks:
        original_drops.append(block.attn.attn_drop)

        # Closure to capture the attention matrix
        captured = []
        class CaptureDrop(torch.nn.Module):
            def forward(self, x):
                captured.append(x.detach())
                return x  # no dropout in eval mode anyway

        block.attn.attn_drop = CaptureDrop()
        block._captured = captured

    with torch.no_grad():
        _ = model(img_tensor)

    for i, block in enumerate(model.blocks):
        if block._captured:
            attn_weights.append(block._captured[0])   # (1, H, 197, 197)
        else:
            attn_weights.append(None)
        block.attn.attn_drop = original_drops[i]      # restore
        del block._captured

    return attn_weights

print('extract_attention_weights defined.')

In [ ]:
# Visualize CLS-token attention from the last block, averaged over heads

fig, axes = plt.subplots(len(sample_images), 3, figsize=(12, 4 * len(sample_images)))

if len(sample_images) == 1:
    axes = [axes]

for row, (pil_img, true_class, true_idx) in enumerate(sample_images):
    img_t = normalize(to_tensor(pil_img)).unsqueeze(0).to(device)

    attn_all = extract_attention_weights(model, img_t)

    # Last block: (1, 12, 197, 197)
    last_attn = attn_all[-1][0]           # (12, 197, 197)

    # CLS attends to patches: row 0, columns 1:
    cls_attn = last_attn[:, 0, 1:]        # (12, 196) — 12 heads
    cls_attn_mean = cls_attn.mean(dim=0)  # (196,) — average over heads
    cls_attn_map = cls_attn_mean.reshape(14, 14).cpu().numpy()

    # Normalize
    cls_attn_map = (cls_attn_map - cls_attn_map.min()) / (cls_attn_map.max() - cls_attn_map.min() + 1e-8)

    overlay = overlay_cam(pil_img, cls_attn_map, colormap='viridis')

    axes[row][0].imshow(pil_img)
    axes[row][0].set_title(f'True: {true_class}', fontsize=10)
    axes[row][0].axis('off')

    axes[row][1].imshow(cls_attn_map, cmap='viridis', vmin=0, vmax=1)
    axes[row][1].set_title('Attention map (last block, mean heads)', fontsize=10)
    axes[row][1].axis('off')

    axes[row][2].imshow(overlay)
    axes[row][2].set_title('Overlay', fontsize=10)
    axes[row][2].axis('off')

plt.suptitle('CLS-Token Attention — Last Block', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'attention_maps.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → assets/attention_maps.png')

In [ ]:
# Per-head attention visualization for the first sample image
pil_img0, true_class0, _ = sample_images[0]
img_t0 = normalize(to_tensor(pil_img0)).unsqueeze(0).to(device)
attn_all0 = extract_attention_weights(model, img_t0)
last_attn0 = attn_all0[-1][0]   # (12, 197, 197)

num_heads = last_attn0.shape[0]
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

for h in range(num_heads):
    head_map = last_attn0[h, 0, 1:].reshape(14, 14).cpu().numpy()
    head_map = (head_map - head_map.min()) / (head_map.max() - head_map.min() + 1e-8)
    axes[h].imshow(head_map, cmap='hot', vmin=0, vmax=1)
    axes[h].set_title(f'Head {h+1}', fontsize=9)
    axes[h].axis('off')

plt.suptitle(f'Per-head CLS attention — last block\nImage: {true_class0}', fontsize=13)
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'attention_heads.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → assets/attention_heads.png')

In [ ]:
# Assertions for attention extraction
attn_test = extract_attention_weights(model, img_t0)

assert len(attn_test) == 12, f'Expected 12 blocks, got {len(attn_test)}'
assert attn_test[0].shape == (1, 12, 197, 197), f'Unexpected shape: {attn_test[0].shape}'

# Each row of attention weights sums to ~1 (softmax)
row_sums = attn_test[-1][0, 0].sum(dim=-1)  # (197,)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4), \
    f'Attention rows do not sum to 1: {row_sums[:5]}'

print('All attention map assertions passed.')

[↑ Back to top](#part-1)

---

<a id="part-4"></a>
## Part 4 — Attention Rollout

**Why rollout?**  
The last-block attention map shows only what the final block attends to — but ViT has 12 blocks, and information flows through residual connections at every layer. Attention rollout (Abnar & Zuidema, 2020) propagates attention through all layers, accounting for the residual skip connections, to produce a more complete map of which patches influenced the final prediction.

**Algorithm:**
1. For each block, add the identity matrix (residual) to the attention matrix, then normalize rows to sum to 1
2. Multiply these matrices in sequence: `rollout = A_L @ A_{L-1} @ ... @ A_1`
3. Row 0 of the result = CLS token's aggregated attention across all layers
4. Discard column 0 (CLS-to-CLS) → reshape 196 → 14×14

In [ ]:
def attention_rollout(attn_weights_list, head_fusion='mean', discard_ratio=0.9):
    """
    Attention rollout across all transformer blocks.

    Args:
        attn_weights_list: list of (1, H, N, N) tensors, one per block
        head_fusion: how to combine heads — 'mean', 'max', or 'min'
        discard_ratio: fraction of lowest-attention patches to zero out (noise reduction)

    Returns:
        rollout_map: (14, 14) numpy array normalized 0–1
    """
    result = None
    N = attn_weights_list[0].shape[-1]  # 197

    for attn in attn_weights_list:
        # attn: (1, H, N, N)
        attn = attn[0]  # (H, N, N)

        if head_fusion == 'mean':
            attn_fused = attn.mean(dim=0)   # (N, N)
        elif head_fusion == 'max':
            attn_fused = attn.max(dim=0).values
        elif head_fusion == 'min':
            attn_fused = attn.min(dim=0).values
        else:
            raise ValueError(f'Unknown head_fusion: {head_fusion}')

        # Optional: zero out lowest attention values
        if discard_ratio > 0:
            flat = attn_fused.flatten()
            threshold_idx = int(flat.shape[0] * discard_ratio)
            threshold = flat.sort().values[threshold_idx]
            attn_fused = torch.where(attn_fused >= threshold,
                                      attn_fused,
                                      torch.zeros_like(attn_fused))

        # Add residual (identity)
        I = torch.eye(N, device=attn_fused.device)
        attn_fused = attn_fused + I

        # Renormalize rows
        attn_fused = attn_fused / attn_fused.sum(dim=-1, keepdim=True)

        # Accumulate via matrix multiply
        if result is None:
            result = attn_fused
        else:
            result = torch.matmul(attn_fused, result)

    # CLS row: result[0, 1:] → (196,) patch contributions
    mask = result[0, 1:].cpu().numpy()   # (196,)
    mask = mask.reshape(14, 14)

    # Normalize
    mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
    return mask

print('attention_rollout defined.')

In [ ]:
# Rollout visualization for all sample images
fig, axes = plt.subplots(len(sample_images), 3, figsize=(12, 4 * len(sample_images)))

if len(sample_images) == 1:
    axes = [axes]

for row, (pil_img, true_class, true_idx) in enumerate(sample_images):
    img_t = normalize(to_tensor(pil_img)).unsqueeze(0).to(device)

    attn_all = extract_attention_weights(model, img_t)
    rollout = attention_rollout(attn_all, head_fusion='mean', discard_ratio=0.9)

    overlay = overlay_cam(pil_img, rollout, colormap='plasma')

    axes[row][0].imshow(pil_img)
    axes[row][0].set_title(f'True: {true_class}', fontsize=10)
    axes[row][0].axis('off')

    axes[row][1].imshow(rollout, cmap='plasma', vmin=0, vmax=1)
    axes[row][1].set_title('Attention rollout (12 blocks)', fontsize=10)
    axes[row][1].axis('off')

    axes[row][2].imshow(overlay)
    axes[row][2].set_title('Overlay', fontsize=10)
    axes[row][2].axis('off')

plt.suptitle('Attention Rollout — All 12 Blocks', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'attention_rollout.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → assets/attention_rollout.png')

In [ ]:
# Assertions for rollout
test_rollout = attention_rollout(attn_all0, head_fusion='mean', discard_ratio=0.9)

assert test_rollout.shape == (14, 14), f'Expected (14,14), got {test_rollout.shape}'
assert 0.0 <= test_rollout.min() and test_rollout.max() <= 1.0, 'Rollout values out of [0,1]'

print('All rollout assertions passed.')

[↑ Back to top](#part-1)

---

<a id="part-5"></a>
## Part 5 — Side-by-Side Export

Final comparison: original image → Grad-CAM → attention map (last block) → rollout.  
Exported to `assets/visualization_comparison.png` for the portfolio.

In [ ]:
# Side-by-side comparison: all three methods, first 3 images
n_show = min(3, len(sample_images))

fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))

if n_show == 1:
    axes = [axes]

gradcam_final = AttentionCAM(model)

for row in range(n_show):
    pil_img, true_class, true_idx = sample_images[row]
    img_t = normalize(to_tensor(pil_img)).unsqueeze(0).to(device)

    # Grad-CAM
    cam, pred_idx, pred_prob = gradcam_final(img_t)

    # Attention (last block)
    attn_all = extract_attention_weights(model, img_t)
    last_attn = attn_all[-1][0][:, 0, 1:].mean(dim=0).reshape(14, 14).cpu().numpy()
    last_attn = (last_attn - last_attn.min()) / (last_attn.max() - last_attn.min() + 1e-8)

    # Rollout
    rollout = attention_rollout(attn_all, head_fusion='mean', discard_ratio=0.9)

    pred_class = CLASSES[pred_idx]
    correct = '✓' if pred_idx == true_idx else '✗'

    # Plot
    axes[row][0].imshow(pil_img)
    axes[row][0].set_title(f'{true_class}\nPred: {pred_class} {correct} {pred_prob:.0%}', fontsize=9)
    axes[row][0].axis('off')

    axes[row][1].imshow(overlay_cam(pil_img, cam, colormap='jet'))
    axes[row][1].set_title('Grad-CAM', fontsize=9)
    axes[row][1].axis('off')

    axes[row][2].imshow(overlay_cam(pil_img, last_attn, colormap='viridis'))
    axes[row][2].set_title('Attention (last block)', fontsize=9)
    axes[row][2].axis('off')

    axes[row][3].imshow(overlay_cam(pil_img, rollout, colormap='plasma'))
    axes[row][3].set_title('Attention rollout', fontsize=9)
    axes[row][3].axis('off')

plt.suptitle('Visualization Comparison — Fine-tuned ViT on SUN397', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'visualization_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → assets/visualization_comparison.png')

gradcam_final.remove_hooks()

In [ ]:
print('\n=== Assets exported ===')
for f in sorted(ASSETS_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:40s} {size_kb:.1f} KB')

[↑ Back to top](#part-1)